# Silverstone 2019 decision-time audit

**Question.** Do the historical records and simulator replay represent Verstappen’s state at the same decision point on laps 12–21?

Imagine making the call just after lap 12. You know his rank and how many times he has already stopped. Whether he stops during lap 13 is the outcome you score later. The model input has three values per lap: completed fraction of the race, previous-lap rank divided by field size, and the number of earlier stops. The binary pit label is stored separately.

Open the notebook from the repository root with its Python 3.11 environment. The workspace’s `START_HERE.md` documents the saved Jolpica feature export and the historical-strategy simulator trace used below.

## Pseudocode

1. Select Verstappen’s historical and simulator rows for laps 12–21. Require one row per lap.
2. Check progress against completed laps divided by 52.
3. Check for stops on laps 13 and 20. Start with zero earlier stops and add each stop to the count beginning the following lap.
4. Convert position fractions to ranks using the 20-car field. Compare the ranks side by side.

In [1]:
from pathlib import Path

from f1_ai.silverstone_audit import audit

results = Path("../offline-results")
historical = results / "jolpica-fresh/local_results/silverstone_2019_jolpica_features.csv"
replay = results / "f1-offline/historical_trace.csv"

comparison = audit(historical, replay)
print(comparison.to_string())

     race_progress  real_position  real_prior_stops  real_pit  sim_position  sim_prior_stops  sim_pit sim_compound
lap                                                                                                               
12        0.211538              4                 0         0             1                0        0             
13        0.230769              4                 0         1             1                0        1           A3
14        0.250000              4                 1         0             1                1        0             
15        0.269231              5                 1         0             6                1        0             
16        0.288462              5                 1         0             6                1        0             
17        0.307692              5                 1         0             6                1        0             
18        0.326923              5                 1         0             5     

## Read the result

- **Lap 13.** The historical input is P4 after lap 12 and the simulator input is P1. Both have zero prior stops and both register a pit. The count becomes one on lap 14.
- **Lap 20.** The historical input is P5 after lap 19 and the simulator input is P3. Both have one prior stop and both register another pit. The count becomes two on lap 21.
- Previous-lap rank matches on lap 18 and differs on the other nine laps. Matching feature definitions let us compare the two sources even when their race states differ.
- A pit counted in its own lap's inputs would leak the outcome into the feature. These saved rows pass the prior-stop timing check.

**For the report**, retain the input definitions, checks, and rank mismatch as evidence about how the simulator differs from the real race. **For the presentation and poster**, use laps 13–14 and 20–21 to show when a stop first becomes part of the next decision's input.

The replay follows scheduled stops. Model prediction and race performance with a learned policy require separate experiments.